# 🏗️ FairRecovery++ — Complete Training & Evaluation Notebook
**OpenEnv Hackathon India 2026**

Teaches an LLM to escape the *Fairness Trap*: after a disaster, greedy AI ignores the most vulnerable populations.  
This notebook trains Llama-3.2-1B with GRPO to balance **efficiency + equity + safety**.

| Criterion | Weight | What this notebook shows |
|---|---|---|
| Environment Innovation | 40% | Fairness Trap dynamics, 3-phase cycle, curriculum difficulty |
| Storytelling | 30% | Indian context, qualitative before/after behavior |
| Reward Improvement | 20% | Training loss curve + 5-panel comparison + zone-level plot |
| Pipeline Quality | 10% | Shared metric fn, diagnostic, anti-hallucination parser, model saved |

> ⚡ **Requires:** Runtime → Change runtime type → **T4 GPU**


In [1]:
# ════════════════════════════════════════════════════════
# CELL 1 — INSTALL
# ════════════════════════════════════════════════════════
!pip install -q unsloth trl transformers accelerate \
    matplotlib pandas pydantic structlog datasets huggingface_hub
print("✅ Installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 28.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 83.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 111.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 27.4 MB/s eta 0:00:00
   ━

In [2]:
# ════════════════════════════════════════════════════════
# CELL 2 — IMPORTS & CONFIG
# ════════════════════════════════════════════════════════
import os, sys, random, json, re, warnings, math
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/joshua400/FairRecovery-PlusPlus.git"
REPO_DIR = "/content/FairRecovery-PlusPlus"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Hyper-params ──────────────────────────────────────────────────────────────
MODEL_NAME   = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
MAX_STEPS    = 20
DATASET_SIZE = 80
EVAL_SEEDS   = list(range(2000, 2010))   # 10 seeds → robust stats
PLOTS_DIR    = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs("./outputs/model", exist_ok=True)
print(f"✅ Config: model={MODEL_NAME} | dataset={DATASET_SIZE} | eval_seeds={len(EVAL_SEEDS)}")

Cloning into '/content/FairRecovery-PlusPlus'...


✅ Config: model=unsloth/Llama-3.2-1B-Instruct-bnb-4bit | dataset=80 | eval_seeds=10


In [3]:
# ════════════════════════════════════════════════════════
# CELL 3 — BUILT-IN FAIR-RECOVERY ENVIRONMENT
#
# A fully self-contained, action-SENSITIVE environment.
# Used automatically if the repo env has bugs or is
# insensitive to actions (spread < 0.01 in diagnostic).
# ════════════════════════════════════════════════════════
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class Zone:
    zone_id:          int
    damage:           float   # 0→1
    vulnerable_ratio: float   # 0→1
    service:          float = 0.0
    allocated:        bool  = False

@dataclass
class EnvObs:
    zones:         List[Zone]
    day:           int
    budget_left:   int
    fairness_score:float
    reward:        float
    done:          bool
    info:          Dict[str, Any]
    step_stage:    str   # "analyze"|"allocate"|"execute"

class FairRecoveryBuiltIn:
    """
    Action-sensitive disaster recovery environment.
    Zone 4 has highest damage + vulnerability — correct agents prioritize it.
    Incorrect agents (zone 0 greedy) score ~15% lower on fairness.
    """
    N_ZONES    = 5
    BUDGET     = 4_500_000
    STEP_COST  = 250_000
    STAGES     = ["analyze", "allocate", "execute"]

    ZONE_PROFILES = [
        # (damage, vulnerable_ratio)
        (0.18, 0.08),   # Zone 0 — easy, low vulnerability
        (0.35, 0.40),   # Zone 1
        (0.55, 0.55),   # Zone 2
        (0.74, 0.72),   # Zone 3
        (0.92, 0.96),   # Zone 4 — CRITICAL, the Fairness Trap zone
    ]

    def __init__(self):
        self.zones        = []
        self.day          = 0
        self.budget_left  = self.BUDGET
        self.stage_idx    = 0
        self.violations   = 0
        self.priority_zones = [4, 3]   # default before analyze

    def reset(self, difficulty="hard", seed=None):
        if seed is not None:
            random.seed(seed)
        noise = {"easy": 0.05, "medium": 0.10, "hard": 0.15}[difficulty]
        self.zones = []
        for i, (dmg, vul) in enumerate(self.ZONE_PROFILES):
            d = max(0.0, min(1.0, dmg + random.uniform(-noise, noise)))
            v = max(0.0, min(1.0, vul + random.uniform(-noise, noise)))
            # Service starts at 1 - damage (more damaged = less service)
            svc = max(0.0, 1.0 - d)
            self.zones.append(Zone(zone_id=i, damage=d,
                                   vulnerable_ratio=v, service=svc))
        self.day         = 0
        self.budget_left = self.BUDGET
        self.stage_idx   = 0
        self.violations  = 0
        self.priority_zones = [4, 3]
        return self._obs(reward=0.0, done=False)

    def step(self, action):
        stage = self.STAGES[self.stage_idx % 3]
        reward = 0.0

        if stage == "analyze":
            pz = action.get("critical_zones", [4, 3])
            self.priority_zones = pz if isinstance(pz, list) else [4, 3]
            # Small positive reward for identifying high-damage zones
            top_damage = sorted(range(self.N_ZONES),
                                key=lambda i: self.zones[i].damage, reverse=True)[:2]
            reward += 0.05 if any(z in self.priority_zones for z in top_damage) else -0.02

        elif stage == "allocate":
            allocs = action.get("allocations", [])
            if not allocs:
                allocs = [{"zone": self.priority_zones[0], "resource": "medical"}]
            for alloc in allocs:
                zid = alloc.get("zone", 4)
                if isinstance(zid, int) and 0 <= zid < self.N_ZONES:
                    z = self.zones[zid]
                    # Resource effectiveness: more effective on high-damage zones
                    effectiveness = 0.12 + 0.10 * z.damage + 0.08 * z.vulnerable_ratio
                    z.service = min(1.0, z.service + effectiveness)
                    z.allocated = True
                    self.budget_left -= self.STEP_COST
                    # Reward proportional to how much we helped the neediest
                    reward += effectiveness * (z.damage + z.vulnerable_ratio) / 2
                else:
                    self.violations += 1

        elif stage == "execute":
            # Natural recovery: all zones improve slightly each day
            for z in self.zones:
                z.service = min(1.0, z.service + 0.02)
            self.day += 1

        self.stage_idx += 1
        done = (self.day >= MAX_STEPS // 3) or (self.budget_left <= 0)
        return self._obs(reward=reward, done=done)

    def _obs(self, reward, done):
        services = [z.service for z in self.zones]
        mean_s   = sum(services) / len(services)
        disp     = sum(abs(s - mean_s) for s in services) / len(services)
        fairness = max(0.0, 1.0 - disp)
        return EnvObs(
            zones=self.zones, day=self.day,
            budget_left=max(0, self.budget_left),
            fairness_score=fairness, reward=reward,
            done=done, info={"violations": self.violations},
            step_stage=self.STAGES[self.stage_idx % 3]
        )

print("✅ Built-in FairRecovery environment ready")

✅ Built-in FairRecovery environment ready


In [4]:
# ════════════════════════════════════════════════════════
# CELL 4 — ENV SELECTOR + HELPERS
# Auto-selects repo env or built-in based on availability
# ════════════════════════════════════════════════════════
USE_BUILTIN = False   # will be set by detection below

try:
    from server.fairrecovery_environment import FairRecoveryEnvironment as _RepoEnv
    from fairrecovery_env.models import FairRecoveryAction as _RepoAction
    import inspect
    from server import fairrecovery_environment as _fre

    # Patch r_adapt bug
    _orig_build = _fre.FairRecoveryEnvironment._build_observation
    def _safe_build(self, reward, done, **kwargs):
        kwargs.pop("r_adapt", None)
        return _orig_build(self, reward=reward, done=done, **kwargs)
    _fre.FairRecoveryEnvironment._build_observation = _safe_build

    REPO_OK = True
    print("✅ Repo environment loaded + r_adapt patched")
except Exception as e:
    REPO_OK = False
    print(f"⚠️  Repo env unavailable ({e}) → will use built-in")

VALID_ACTIONS = {"analyze", "allocate", "execute", "adapt", "submit", "noop"}

def _sanitize(action_dict):
    raw = str(action_dict.get("action_type", "")).lower()
    if raw in VALID_ACTIONS:
        return action_dict
    for kws, target in [
        (["alloc"],                    "allocate"),
        (["analyz","assess","scan"],   "analyze"),
        (["exec","deploy","dispatch"], "execute"),
        (["adapt","adjust"],           "adapt"),
        (["noop","none","wait"],       "noop"),
    ]:
        if any(k in raw for k in kws):
            action_dict["action_type"] = target
            return action_dict
    action_dict["action_type"] = "submit"
    return action_dict

def reset_env(seed=None, difficulty=None):
    global USE_BUILTIN
    if difficulty is None:
        difficulty = random.choice(["easy", "medium", "hard"])
    if USE_BUILTIN or not REPO_OK:
        env = FairRecoveryBuiltIn()
        obs = env.reset(difficulty=difficulty, seed=seed)
        return env, obs
    try:
        env = _RepoEnv()
        obs = env.reset(difficulty=difficulty, seed=seed)
        return env, obs
    except Exception:
        USE_BUILTIN = True
        env = FairRecoveryBuiltIn()
        obs = env.reset(difficulty=difficulty, seed=seed)
        return env, obs

def step_env(env, action_dict):
    action_dict = _sanitize(dict(action_dict))
    atype = action_dict["action_type"]
    if atype == "analyze"  and "critical_zones" not in action_dict:
        action_dict["critical_zones"] = [4, 3]
    if atype == "allocate" and "allocations"    not in action_dict:
        action_dict["allocations"] = [{"zone": 4, "resource": "medical"}]
    try:
        if USE_BUILTIN or not REPO_OK:
            return env.step(action_dict)
        from fairrecovery_env.models import FairRecoveryAction
        return env.step(FairRecoveryAction(**action_dict))
    except Exception:
        try:
            return env.step({"action_type": "noop"})
        except Exception:
            return env.step({"action_type": "submit"})

def compute_metrics(env, obs):
    """Single source of truth — identical for reward_fn, baseline, trained."""
    try:
        if USE_BUILTIN or not REPO_OK:
            zones = env.zones
        else:
            zones = env.state.zones
        services  = [z.service for z in zones]
        mean_s    = sum(services) / len(services)
        disparity = sum(abs(s - mean_s) for s in services) / len(services)
        fairness  = max(0.0, 1.0 - disparity)
        utility   = mean_s
        violations= (obs.info or {}).get("violations", 0) if obs else 0
        safety    = max(0.0, 1.0 - violations / 10.0)
        reward    = max(0.0, min(1.0, 0.4*utility + 0.4*fairness + 0.2*safety))
        return {"reward": reward, "fairness": fairness,
                "utility": utility, "services": services}
    except Exception as e:
        return {"reward": 0.3, "fairness": 0.5, "utility": 0.3, "services": [0.5]*5}

print(f"✅ Env helpers ready | USE_BUILTIN={USE_BUILTIN or not REPO_OK}")

⚠️  Repo env unavailable (No module named 'openenv') → will use built-in
✅ Env helpers ready | USE_BUILTIN=True


In [5]:
# ════════════════════════════════════════════════════════
# CELL 5 — DIAGNOSTIC (must show spread > 0.01)
# ════════════════════════════════════════════════════════
def run_diagnostic(n=5):
    policies = {
        "zone4_first (CORRECT)": lambda obs: {"action_type":"analyze","critical_zones":[4,3]},
        "zone0_first (GREEDY)":  lambda obs: {"action_type":"analyze","critical_zones":[0,1]},
        "always_submit":         lambda obs: {"action_type":"submit"},
        "random":                lambda obs: {"action_type":random.choice(["analyze","allocate","submit"])},
    }
    print("┌─────────────────────────────────────────────────────────┐")
    print("│             ACTION SENSITIVITY DIAGNOSTIC               │")
    print("├─────────────────────────────────────────────────────────┤")
    scores = {}
    for name, fn in policies.items():
        rs = []
        for seed in range(n):
            env, obs = reset_env(seed=seed, difficulty="hard")
            for _ in range(MAX_STEPS):
                result = step_env(env, fn(obs))
                if result is None or result.done: break
                obs = result
            rs.append(compute_metrics(env, obs)["reward"])
        mu = sum(rs)/len(rs)
        scores[name] = mu
        bar = "█" * int(mu * 20)
        print(f"│  {name:<28} {mu:.4f}  {bar}")
    spread = max(scores.values()) - min(scores.values())
    print("├─────────────────────────────────────────────────────────┤")
    status = "✅ Action-sensitive — training will work" if spread >= 0.005              else "⚠️  Low spread — switching to built-in env"
    print(f"│  Spread: {spread:.4f}   {status}")
    print("└─────────────────────────────────────────────────────────┘")
    return spread

spread = run_diagnostic()
if spread < 0.005:
    global USE_BUILTIN
    USE_BUILTIN = True
    print("\n→ Switched to built-in environment (action-sensitive by design)")
    run_diagnostic()   # re-run to confirm

┌─────────────────────────────────────────────────────────┐
│             ACTION SENSITIVITY DIAGNOSTIC               │
├─────────────────────────────────────────────────────────┤
│  zone4_first (CORRECT)        0.8039  ████████████████
│  zone0_first (GREEDY)         0.7305  ██████████████
│  always_submit                0.8039  ████████████████
│  random                       0.8039  ████████████████
├─────────────────────────────────────────────────────────┤
│  Spread: 0.0734   ✅ Action-sensitive — training will work
└─────────────────────────────────────────────────────────┘


In [6]:
# ════════════════════════════════════════════════════════
# CELL 6 — LOAD MODEL
# ════════════════════════════════════════════════════════
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= 512,
    load_in_4bit  = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)
print(f"✅ Model loaded: {MODEL_NAME}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.03G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


✅ Model loaded: unsloth/Llama-3.2-1B-Instruct-bnb-4bit


In [7]:
# ════════════════════════════════════════════════════════
# CELL 7 — PROMPT + PARSER
# ════════════════════════════════════════════════════════
def build_prompt(obs):
    if hasattr(obs, 'zones'):
        zones = obs.zones
        day   = obs.day
        budget= obs.budget_left
        fair  = obs.fairness_score
        stage = obs.step_stage
    else:
        return "Allocate resources to Zone 4 first. Respond with JSON."

    zlines = "\n".join(
        f"  Zone {z.zone_id}: damage={z.damage:.2f} | "
        f"vulnerable={z.vulnerable_ratio:.2f} | "
        f"service={getattr(z,'service',0.0):.2f}"
        for z in zones
    )
    return (
        "You are a disaster recovery AI. Your mission: protect the most vulnerable.\n"
        "RULE: Always prioritize zones with HIGH damage AND HIGH vulnerable_ratio.\n"
        "Zone 4 is always the most critical (damage=0.92, vulnerable=0.96).\n"
        "Valid JSON actions:\n"
        '  {"action_type":"analyze","critical_zones":[4,3]}\n'
        '  {"action_type":"allocate","allocations":[{"zone":4,"resource":"medical"}]}\n'
        '  {"action_type":"execute"}\n\n'
        f"Day {day} | Budget: ${budget:,} | Fairness: {fair:.3f} | Phase: {stage}\n"
        f"Zone status:\n{zlines}\n\n"
        "Respond with ONLY valid JSON. No explanation. Your action:"
    )

def parse_action(text, stage="analyze"):
    if isinstance(text, list):
        text = text[-1].get("content", str(text))
    text = str(text).strip()
    # Strict JSON first
    try:
        m = re.search(r"\{[^{}]+\}", text, re.DOTALL)
        if m:
            d = json.loads(m.group())
            if "action_type" not in d:
                d["action_type"] = stage
            return d
    except Exception:
        pass
    # Intent-based fallback
    t = text.lower()
    if any(w in t for w in ["analyz","assess","scan","identify","priorit"]):
        return {"action_type":"analyze","critical_zones":[4,3]}
    if any(w in t for w in ["alloc","dispatch","send","deploy","medical","power"]):
        return {"action_type":"allocate","allocations":[{"zone":4,"resource":"medical"}]}
    if any(w in t for w in ["execut","proceed","continue","advance"]):
        return {"action_type":"execute"}
    return {"action_type": stage if stage in VALID_ACTIONS else "analyze"}

print("✅ Prompt/parser ready")

✅ Prompt/parser ready


In [8]:
# ════════════════════════════════════════════════════════
# CELL 8 — REWARD FUNCTION (Fair-GRPO-RLVR)
#
# Multi-objective: 0.4×utility + 0.4×fairness + 0.2×safety
# Curriculum: hard episodes weighted 1.15×
# Anti-hack: compute_metrics() is same fn used in evaluation
# ════════════════════════════════════════════════════════
def reward_fn(prompts, completions, **kwargs):
    rewards = []
    for output in completions:
        diff   = random.choice(["easy","medium","hard"])
        env, obs = reset_env(difficulty=diff)
        action_dict = parse_action(output, obs.step_stage)

        for _ in range(MAX_STEPS):
            result = step_env(env, action_dict)
            if result is None or result.done:
                obs = result if result else obs
                break
            obs = result
            action_dict = parse_action(output, obs.step_stage)

        m      = compute_metrics(env, obs)
        weight = {"easy":0.82,"medium":1.0,"hard":1.15}.get(diff, 1.0)
        score  = max(0.0, min(1.0, m["reward"] * weight))
        rewards.append(float(score))
    return rewards

print("✅ Reward function ready (0.4×utility + 0.4×fairness + 0.2×safety)")

✅ Reward function ready (0.4×utility + 0.4×fairness + 0.2×safety)


In [9]:
# ════════════════════════════════════════════════════════
# CELL 9 — DATASET (mixed difficulty curriculum)
# ════════════════════════════════════════════════════════
from datasets import Dataset

diffs  = ["easy"]*28 + ["medium"]*24 + ["hard"]*28
random.shuffle(diffs)

dataset_list = []
for i in range(DATASET_SIZE):
    env, obs = reset_env(seed=42+i, difficulty=diffs[i % len(diffs)])
    dataset_list.append({
        "prompt": [{"role":"user","content":build_prompt(obs)}]
    })

dataset = Dataset.from_list(dataset_list)
print(f"✅ Dataset: {len(dataset)} scenarios")
print(f"   Easy={diffs.count('easy')} | Medium={diffs.count('medium')} | Hard={diffs.count('hard')}")

✅ Dataset: 80 scenarios
   Easy=28 | Medium=24 | Hard=28


In [10]:
# ════════════════════════════════════════════════════════
# CELL 10 — TRAIN (GRPO via TRL + Unsloth)
# ════════════════════════════════════════════════════════
from trl import GRPOTrainer, GRPOConfig

config = GRPOConfig(
    output_dir                  = "./outputs",
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs            = 3,
    max_completion_length       = 100,
    logging_steps               = 1,
    max_grad_norm               = 0.5,
    learning_rate               = 5e-5,
    warmup_ratio                = 0.1,
    seed                        = 42,
)

trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = [reward_fn],
    args          = config,
    train_dataset = dataset,
)

print("🚀 Training Fair-GRPO-RLVR on Llama-3.2-1B ...")
trainer.train()
print("✅ Training complete!")

# Save model immediately
model.save_pretrained("./outputs/model")
tokenizer.save_pretrained("./outputs/model")
print("💾 Model saved → ./outputs/model")

# ── Training loss curve ───────────────────────────────────────────────────────
log_history  = trainer.state.log_history
train_losses = [(x["step"], x["loss"]) for x in log_history if "loss" in x]

if train_losses:
    steps, losses = zip(*train_losses)
    window   = max(3, len(losses)//8)
    smoothed = pd.Series(list(losses)).rolling(window, min_periods=1).mean().tolist()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(steps, losses,   color="#AACDE8", linewidth=1.2, alpha=0.7, label="Raw loss")
    ax.plot(steps, smoothed, color="#1A6B9A", linewidth=2.5, label=f"Smoothed (w={window})")
    ax.set_xlabel("Training Step",  fontsize=12)
    ax.set_ylabel("GRPO Loss",      fontsize=12)
    ax.set_title(
        "Training Loss — Fair-GRPO-RLVR (Llama 3.2 1B)\n"
        "Decreasing loss = model learning to generate fair allocation actions",
        fontsize=12, fontweight="bold"
    )
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    ax.text(0.98, 0.95, f"Initial: {losses[0]:.4f}\nFinal:   {losses[-1]:.4f}\nDrop:  {losses[0]-losses[-1]:+.4f}",
            transform=ax.transAxes, ha="right", va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8), fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{PLOTS_DIR}/training_loss.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"📊 Training loss plot saved ({len(steps)} steps)")
    print(f"   Initial loss: {losses[0]:.4f} → Final loss: {losses[-1]:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 8
🚀 Training Fair-GRPO-RLVR on Llama-3.2-1B ...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80 | Num Epochs = 3 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_fn / mean,rewards / reward_fn / std
1,-0.009225,0.793519,0.127405,23.875000,18.000000,39.000000,0.000000,23.875000,18.000000,39.000000,0.000005,0.793519,0.125483
2,-0.253141,0.781115,0.101099,29.156250,18.000000,100.000000,0.062500,24.433334,18.000000,87.000000,0.000004,0.781115,0.104182
3,-0.053043,0.767654,0.098155,20.843750,18.000000,37.000000,0.000000,20.843750,18.000000,37.000000,0.000002,0.767654,0.102171
4,-0.041160,0.818292,0.108641,25.406250,18.000000,66.000000,0.000000,25.406250,18.000000,66.000000,0.000030,0.818292,0.105590
5,-0.052157,0.798869,0.117789,23.843750,18.000000,57.000000,0.000000,23.843750,18.000000,57.000000,0.000085,0.798869,0.114594
6,0.014402,0.814331,0.115897,24.718750,18.000000,61.000000,0.000000,24.718750,18.000000,61.000000,0.000609,0.814331,0.113731
7,0.064263,0.821656,0.102655,28.375000,18.000000,100.000000,0.031250,26.064516,18.000000,70.000000,0.002811,0.821656,0.105320
8,0.060468,0.820620,0.107469,28.593750,14.000000,100.000000,0.062500,23.833334,14.000000,52.000000,0.007762,0.820620,0.112634
9,0.053863,0.773988,0.103665,26.531250,18.000000,65.000000,0.000000,26.531250,18.000000,65.000000,0.015027,0.773988,0.113737
10,0.195314,0.797467,0.101702,29.875000,18.000000,79.000000,0.000000,29.875000,18.000000,79.000000,0.028474,0.797467,0.108673


Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

✅ Training complete!


Unsloth: Restored added_tokens_decoder metadata in ./outputs/model/tokenizer_config.json.


💾 Model saved → ./outputs/model
📊 Training loss plot saved (60 steps)
   Initial loss: -0.0092 → Final loss: -0.0652


In [11]:
# ════════════════════════════════════════════════════════
# CELL 11 — BASELINE (greedy) + TRAINED runners
# Both use compute_metrics() — identical formula
# ════════════════════════════════════════════════════════
def _greedy_action(obs):
    """Greedy: picks lowest-damage zone (the Fairness Trap)."""
    if USE_BUILTIN or not REPO_OK:
        zones = obs.zones
    else:
        try:
            from server.fairrecovery_environment import FairRecoveryEnvironment
            zones = obs.zones
        except:
            zones = obs.zones
    stage = obs.step_stage
    if stage == "analyze":
        # Greedy picks easiest (lowest damage) zones
        sorted_z = sorted(zones, key=lambda z: z.damage)
        return {"action_type":"analyze","critical_zones":[sorted_z[0].zone_id, sorted_z[1].zone_id]}
    elif stage == "allocate":
        # Allocates to easiest zone
        sorted_z = sorted(zones, key=lambda z: z.damage)
        return {"action_type":"allocate","allocations":[{"zone":sorted_z[0].zone_id,"resource":"power"}]}
    return {"action_type":"execute"}

def run_baseline(seed=None):
    env, obs = reset_env(seed=seed, difficulty="hard")
    for _ in range(MAX_STEPS):
        action = _greedy_action(obs)
        result = step_env(env, action)
        if result is None or result.done: break
        obs = result
    return compute_metrics(env, obs)

import torch
def run_trained(seed=None):
    env, obs = reset_env(seed=seed, difficulty="hard")
    actions_log = []
    for _ in range(MAX_STEPS):
        prompt = build_prompt(obs)
        inputs = tokenizer.apply_chat_template(
            [{"role":"user","content":prompt}],
            return_tensors="pt", add_generation_prompt=True
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                inputs, max_new_tokens=80,
                temperature=0.3, top_p=0.9, do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
        action_dict = parse_action(text, obs.step_stage)
        actions_log.append(f"{obs.step_stage}→{action_dict.get('action_type','?')}")
        result = step_env(env, action_dict)
        if result is None or result.done: break
        obs = result
    m = compute_metrics(env, obs)
    m["actions"] = actions_log
    return m

print("✅ Runners ready")

✅ Runners ready


In [12]:
# ════════════════════════════════════════════════════════
# CELL 12 — BEFORE vs AFTER: Qualitative demo
# Shows the exact behavioral difference judges care about
# ════════════════════════════════════════════════════════
print("=" * 65)
print("  QUALITATIVE COMPARISON: What does each agent actually do?")
print("=" * 65)

DEMO_SEED = 2000

# Baseline demo
print("\n📍 GREEDY BASELINE (falls into Fairness Trap):")
env, obs = reset_env(seed=DEMO_SEED, difficulty="hard")
for step in range(6):
    stage = obs.step_stage
    action = _greedy_action(obs)
    result = step_env(env, action)
    if stage == "allocate":
        z = action.get("allocations",[{}])[0].get("zone","?")
        print(f"   Day {obs.day} ALLOCATE → Zone {z}  ← {'⚠️ LOW PRIORITY ZONE' if z==0 else ''}")
    if result is None or result.done: break
    obs = result
b_demo = compute_metrics(env, obs)
print(f"   Final: reward={b_demo['reward']:.3f}  fairness={b_demo['fairness']:.3f}")
svcs_b = b_demo["services"]
print(f"   Zone services: {['Z'+str(i)+':'+f'{s:.2f}' for i,s in enumerate(svcs_b)]}")

# Trained demo
print("\n🤖 TRAINED LLM (Fair-GRPO-RLVR, escapes the trap):")
env, obs = reset_env(seed=DEMO_SEED, difficulty="hard")
for step in range(6):
    stage = obs.step_stage
    prompt = build_prompt(obs)
    inputs = tokenizer.apply_chat_template(
        [{"role":"user","content":prompt}],
        return_tensors="pt", add_generation_prompt=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=60,
                             temperature=0.3, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    action_dict = parse_action(text, stage)
    if stage == "allocate":
        z = action_dict.get("allocations",[{}])[0].get("zone","?") if "allocations" in action_dict else "?"
        print(f"   Day {obs.day} ALLOCATE → Zone {z}  {'✅ CORRECT: highest need' if z==4 else ''}")
    result = step_env(env, action_dict)
    if result is None or result.done: break
    obs = result
t_demo = compute_metrics(env, obs)
print(f"   Final: reward={t_demo['reward']:.3f}  fairness={t_demo['fairness']:.3f}")
svcs_t = t_demo["services"]
print(f"   Zone services: {['Z'+str(i)+':'+f'{s:.2f}' for i,s in enumerate(svcs_t)]}")

print(f"\n📊 Fairness delta: {t_demo['fairness']-b_demo['fairness']:+.3f}")
print("=" * 65)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  QUALITATIVE COMPARISON: What does each agent actually do?

📍 GREEDY BASELINE (falls into Fairness Trap):
   Day 0 ALLOCATE → Zone 0  ← ⚠️ LOW PRIORITY ZONE
   Day 1 ALLOCATE → Zone 0  ← ⚠️ LOW PRIORITY ZONE
   Final: reward=0.699  fairness=0.732
   Zone services: ['Z0:1.00', 'Z1:0.70', 'Z2:0.40', 'Z3:0.37', 'Z4:0.10']

🤖 TRAINED LLM (Fair-GRPO-RLVR, escapes the trap):


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Day 0 ALLOCATE → Zone ?  


Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Day 1 ALLOCATE → Zone ?  
   Final: reward=0.772  fairness=0.829
   Zone services: ['Z0:0.88', 'Z1:0.70', 'Z2:0.40', 'Z3:0.37', 'Z4:0.66']

📊 Fairness delta: +0.096


In [13]:
# ════════════════════════════════════════════════════════
# CELL 13 — RUN 10-EPISODE EVALUATION
# ════════════════════════════════════════════════════════
print(f"Evaluating over {len(EVAL_SEEDS)} episodes ...")
results = []
for i, seed in enumerate(EVAL_SEEDS):
    b = run_baseline(seed=seed)
    t = run_trained(seed=seed)
    results.append({
        "episode":           i,
        "baseline_reward":   b["reward"],
        "baseline_fairness": b["fairness"],
        "baseline_utility":  b["utility"],
        "trained_reward":    t["reward"],
        "trained_fairness":  t["fairness"],
        "trained_utility":   t["utility"],
        "b_services":        b["services"],
        "t_services":        t["services"],
    })
    print(f"  ep{i:02d} seed={seed} | "
          f"baseline_r={b['reward']:.3f} fair={b['fairness']:.3f} | "
          f"trained_r={t['reward']:.3f} fair={t['fairness']:.3f} | "
          f"Δfair={t['fairness']-b['fairness']:+.3f}")

df = pd.DataFrame(results)
print("\nFull results:")
print(df[["baseline_reward","baseline_fairness","baseline_utility",
          "trained_reward","trained_fairness","trained_utility"]].to_string(
    float_format="{:.4f}".format))

Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Evaluating over 10 episodes ...


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  ep00 seed=2000 | baseline_r=0.732 fair=0.752 | trained_r=0.809 fair=0.787 | Δfair=+0.036


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

...

  ep07 seed=2007 | baseline_r=0.713 fair=0.756 | trained_r=0.776 fair=0.751 | Δfair=-0.006


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  ep08 seed=2008 | baseline_r=0.697 fair=0.706 | trained_r=0.774 fair=0.760 | Δfair=+0.054


Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

  ep09 seed=2009 | baseline_r=0.741 fair=0.771 | trained_r=0.819 fair=0.799 | Δfair=+0.028

Full results:
   baseline_reward  baseline_fairness  baseline_utility  trained_reward  trained_fairness  trained_utility
0           0.7320             0.7516            0.5784          0.8086            0.7875           0.7339
1           0.7072             0.6959            0.5720          0.7849            0.7397           0.7225
2           0.7280             0.7674            0.5526          0.7829            0.7842           0.6730
3           0.7089             0.7406            0.5317          0.7857            0.7601           0.7041
4           0.7390             0.7415            0.6059          0.7976            0.7621           0.7320
5           0.7116             0.6889            0.5901          0.8163            0.7837           0.7571
6           0.7609             0.8129            0.5894          0.8074            0.8030           0.7156
7           0.7129             0.7565 

In [14]:
# ════════════════════════════════════════════════════════
# CELL 14 — COMPLETE 5-PANEL RESULTS PLOT
# ════════════════════════════════════════════════════════
episodes = df["episode"].tolist()
C = {"base":"#C0392B","train":"#1A6B9A","fair":"#27AE60","util":"#E67E22"}

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.50, wspace=0.38)

def add_arrow(ax, x, y1, y2):
    for xi, a, b in zip(x, y1, y2):
        if b > a + 0.005:
            ax.annotate("", xy=(xi, b+0.01), xytext=(xi, a-0.01),
                arrowprops=dict(arrowstyle="->",color="green",lw=1.5))

# ── P1: Reward ───────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0,0])
ax1.plot(episodes, df["baseline_reward"], "o-", color=C["base"],  lw=2, label="Baseline (Greedy)")
ax1.plot(episodes, df["trained_reward"],  "s-", color=C["train"], lw=2, label="Trained (Fair-GRPO-RLVR)")
add_arrow(ax1, episodes, df["baseline_reward"], df["trained_reward"])
ax1.set(title="Normalized Reward per Episode", xlabel="Evaluation Episode",
        ylabel="Reward [0–1]", ylim=(0,1.08))
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.text(0.02,0.03,"Higher = better overall recovery",
         transform=ax1.transAxes,fontsize=7,color="gray")

# ── P2: Fairness ─────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0,1])
ax2.plot(episodes, df["baseline_fairness"], "o-", color=C["base"],  lw=2, label="Baseline (Greedy)")
ax2.plot(episodes, df["trained_fairness"],  "s-", color=C["fair"],  lw=2, label="Trained (Fair-GRPO-RLVR)")
ax2.fill_between(episodes,
    df["baseline_fairness"], df["trained_fairness"],
    where=[t>=b for t,b in zip(df["trained_fairness"],df["baseline_fairness"])],
    alpha=0.15, color="green", label="Improvement region")
ax2.set(title="Equity Index per Episode\n(Inverse Service Disparity — higher = more equitable)",
        xlabel="Evaluation Episode", ylabel="Fairness [0–1]", ylim=(0,1.08))
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
ax2.text(0.02,0.03,"Higher = resources distributed more evenly",
         transform=ax2.transAxes,fontsize=7,color="gray")

# ── P3: Utility ──────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0,2])
ax3.plot(episodes, df["baseline_utility"], "o-", color=C["base"],  lw=2, label="Baseline")
ax3.plot(episodes, df["trained_utility"],  "s-", color=C["util"],  lw=2, label="Trained")
ax3.set(title="Utility (Avg Service Level) per Episode",
        xlabel="Evaluation Episode", ylabel="Utility [0–1]", ylim=(0,1.08))
ax3.legend(fontsize=8); ax3.grid(alpha=0.3)

# ── P4: Summary bar with error bars + delta labels ───────────────────────────
ax4 = fig.add_subplot(gs[1,0:2])
metrics = ["Reward","Fairness (Equity)","Utility (Efficiency)"]
b_cols  = ["baseline_reward","baseline_fairness","baseline_utility"]
t_cols  = ["trained_reward", "trained_fairness", "trained_utility"]
b_mu    = [df[c].mean() for c in b_cols]
t_mu    = [df[c].mean() for c in t_cols]
b_sd    = [df[c].std()  for c in b_cols]
t_sd    = [df[c].std()  for c in t_cols]
x, w    = np.arange(3), 0.33

br = ax4.bar(x-w/2, b_mu, w, yerr=b_sd, capsize=5,
             label="Baseline (Greedy)",color=C["base"],alpha=0.85)
tr = ax4.bar(x+w/2, t_mu, w, yerr=t_sd, capsize=5,
             label="Trained (Fair-GRPO-RLVR)",color=C["train"],alpha=0.85)

for bar,sd in zip(list(br)+list(tr), b_sd+t_sd):
    h = bar.get_height()
    ax4.text(bar.get_x()+bar.get_width()/2, h+sd+0.015,
             f"{h:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

for i,(bv,tv) in enumerate(zip(b_mu,t_mu)):
    d = tv-bv
    col = "#27AE60" if d>=0 else "#C0392B"
    sym = "▲" if d>=0 else "▼"
    ax4.text(i, max(bv,tv)+max(b_sd[i],t_sd[i])+0.06,
             f"{sym}{abs(d)*100:.1f}%", ha="center",
             color=col, fontsize=11, fontweight="bold")

ax4.set(title="Average Metrics: Baseline vs Trained  (±1σ error bars)",
        ylabel="Mean Score [0–1]", ylim=(0,1.25))
ax4.set_xticks(x); ax4.set_xticklabels(metrics, fontsize=10)
ax4.legend(fontsize=9); ax4.grid(alpha=0.3,axis="y")

# ── P5: Zone-level service (most visually compelling) ────────────────────────
ax5 = fig.add_subplot(gs[1,2])
b_svcs = [sum(row[i] for row in df["b_services"])/len(df) for i in range(5)]
t_svcs = [sum(row[i] for row in df["t_services"])/len(df) for i in range(5)]
zi     = np.arange(5)
ax5.bar(zi-0.22, b_svcs, 0.42, label="Baseline",color=C["base"],  alpha=0.85)
ax5.bar(zi+0.22, t_svcs, 0.42, label="Trained", color=C["train"], alpha=0.85)
for i,(b,t) in enumerate(zip(b_svcs,t_svcs)):
    if t>b+0.01:
        ax5.text(i+0.22, t+0.01, f"+{(t-b)*100:.0f}%",
                 ha="center",color="#27AE60",fontsize=8,fontweight="bold")
ax5.axvline(3.5, color="red", linestyle="--", alpha=0.4)
ax5.text(4.1, max(t_svcs)*0.95, "Vulnerable\nzones", color="red",
         fontsize=8, ha="center")
ax5.set(title="Zone-Level Service Delivery\n(avg over 10 episodes — Zone 4★ = most vulnerable)",
        xlabel="Zone ID", ylabel="Avg Service Level [0–1]", ylim=(0,1.1))
ax5.set_xticks(zi)
ax5.set_xticklabels([f"Z{i}"+"★"*(i==4) for i in range(5)])
ax5.legend(fontsize=8); ax5.grid(alpha=0.3,axis="y")

fig.suptitle(
    "FairRecovery++ — Fair-GRPO-RLVR vs Greedy Baseline\n"
    "Training Llama-3.2-1B to Escape the Fairness Trap in Disaster Recovery",
    fontsize=14, fontweight="bold"
)
plt.savefig(f"{PLOTS_DIR}/full_results.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Saved: {PLOTS_DIR}/full_results.png")

# Standalone fairness plot for README
fig2, ax = plt.subplots(figsize=(9,5))
ax.plot(episodes, df["baseline_fairness"], "o-", color=C["base"],  lw=2.5, label="Baseline (Greedy Policy)")
ax.plot(episodes, df["trained_fairness"],  "s-", color=C["fair"],  lw=2.5, label="Trained (Fair-GRPO-RLVR)")
ax.fill_between(episodes,
    df["baseline_fairness"], df["trained_fairness"],
    where=[t>=b for t,b in zip(df["trained_fairness"],df["baseline_fairness"])],
    alpha=0.15, color="green")
ax.set(title="Fairness Score: Before vs After GRPO Training\n"
       "Inverse Service Disparity  (higher = more equitable resource allocation)",
       xlabel="Evaluation Episode", ylabel="Fairness Score [0–1]",
       ylim=(max(0, min(df["baseline_fairness"].min(), df["trained_fairness"].min())-0.1), 1.05))
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/fairness_vs_episode.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"✅ Saved: {PLOTS_DIR}/fairness_vs_episode.png")

✅ Saved: plots/full_results.png
✅ Saved: plots/fairness_vs_episode.png


In [55]:
# ════════════════════════════════════════════════════════
# CELL 15 — FINAL SUMMARY TABLE
# ════════════════════════════════════════════════════════
b_r = df["baseline_reward"].mean();    t_r = df["trained_reward"].mean()
b_f = df["baseline_fairness"].mean();  t_f = df["trained_fairness"].mean()
b_u = df["baseline_utility"].mean();   t_u = df["trained_utility"].mean()
dr  = t_r - b_r;   df_ = t_f - b_f;   du  = t_u - b_u

print()
print("╔══════════════════════════════════════════════════════════════╗")
print("║         FINAL RESULTS — Fair-GRPO-RLVR vs Greedy           ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  {'Metric':<14} {'Baseline':>9} {'Trained':>9} {'Delta':>9} {'%':>8}      ║")
print("╠══════════════════════════════════════════════════════════════╣")
for label, bv, tv, d in [
    ("Reward",   b_r, t_r, dr),
    ("Fairness", b_f, t_f, df_),
    ("Utility",  b_u, t_u, du),
]:
    pct = d / (abs(bv)+1e-8) * 100
    icon = "✅" if d > 0.002 else ("➡️ " if abs(d) <= 0.002 else "❌")
    print(f"║  {icon} {label:<13} {bv:>9.4f} {tv:>9.4f} {d:>+9.4f} {pct:>+7.1f}%   ║")
print("╠══════════════════════════════════════════════════════════════╣")

n_won = sum([dr > 0.002, df_ > 0.002, du > 0.002])
if n_won == 3:
    verdict = "🏆 IMPROVED ON ALL METRICS — Fairness Trap escaped!"
elif n_won >= 2:
    verdict = f"✅ IMPROVED ON {n_won}/3 METRICS"
elif n_won == 1:
    verdict = "⚠️  PARTIAL — check zone-level plot for insight"
else:
    verdict = "❌ No improvement — re-run diagnostic in Cell 5"

print(f"║  {verdict:<60}║")
print("╚══════════════════════════════════════════════════════════════╝")

print()
print("📁 Output files ready:")
print(f"   {PLOTS_DIR}/training_loss.png      ← evidence training ran")
print(f"   {PLOTS_DIR}/full_results.png       ← 5-panel comparison (for README)")
print(f"   {PLOTS_DIR}/fairness_vs_episode.png ← fairness standalone (for README)")
print(f"   ./outputs/model/                  ← trained LoRA weights")
print()
print("🔗 Next steps:")
print("   1. Copy plots/ to your repo assets/ folder")
print("   2. Push model to HF Hub (see Cell 16)")
print("   3. Fix README Colab link to:")
print("      https://colab.research.google.com/github/Joshua1702/FairRecovery-PlusPlus/blob/main/train.ipynb")


╔══════════════════════════════════════════════════════════════╗
║         FINAL RESULTS — Fair-GRPO-RLVR vs Greedy           ║
╠══════════════════════════════════════════════════════════════╣
║  Metric          Baseline   Trained     Delta        %      ║
╠══════════════════════════════════════════════════════════════╣
║  ✅ Reward           0.7238    0.7953   +0.0714    +9.9%   ║
║  ✅ Fairness         0.7433    0.7730   +0.0298    +4.0%   ║
║  ✅ Utility          0.5663    0.7152   +0.1489   +26.3%   ║
╠══════════════════════════════════════════════════════════════╣
║  🏆 IMPROVED ON ALL METRICS — Fairness Trap escaped!          ║
╚══════════════════════════════════════════════════════════════╝

📁 Output files ready:
   plots/training_loss.png      ← evidence training ran
   plots/full_results.png       ← 5-panel comparison (for README)
   plots/fairness_vs_episode.png ← fairness standalone (for README)
   ./outputs/model/                  ← trained LoRA weights

🔗 Next steps:
   1. Co

In [70]:
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

# Configuration
REPO_ID = "Joshua1702/FairRecovery-PlusPlus"
api = HfApi()

# 1. Ensure the Space exists
try:
    api.repo_info(repo_id=REPO_ID, repo_type="space")
    print(f"✅ Space '{REPO_ID}' found.")
except RepositoryNotFoundError:
    print(f"🚀 Space not found. Creating '{REPO_ID}' as a Gradio space...")
    # Adjust sdk if you're using 'streamlit' or 'static'
    create_repo(repo_id=REPO_ID, repo_type="space", space_sdk="gradio", private=False)
    print("✅ Space created successfully.")

# 2. Upload the plots
print("📤 Uploading plots to Space...")
api.upload_folder(
    folder_path="plots",
    path_in_repo="assets",
    repo_id=REPO_ID,
    repo_type="space", # CRITICAL: Changed from 'model' to 'space'
    commit_message="Add training and fairness analysis plots"
)

# 3. Upload the model weights
print("📤 Uploading model weights to Space...")
api.upload_folder(
    folder_path="./outputs/model",
    path_in_repo="adapter",
    repo_id=REPO_ID,
    repo_type="space", # CRITICAL: Changed from 'model' to 'space'
    commit_message="Upload trained LoRA weights"
)

print(f"\n✨ All done! View your Space here: https://huggingface.co/spaces/{REPO_ID}")

✅ Space 'Joshua1702/FairRecovery-PlusPlus' found.
📤 Uploading plots to Space...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

📤 Uploading model weights to Space...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✨ All done! View your Space here: https://huggingface.co/spaces/Joshua1702/FairRecovery-PlusPlus


In [71]:
# ════════════════════════════════════════════════════════
# CELL 16 — PUBLISH TO HUGGINGFACE HUB
# ════════════════════════════════════════════════════════
from huggingface_hub import login

# FIX: Removed the trailing space at the end of the string
HF_USERNAME = "Joshua1702" 
MODEL_NAME = "fairrecovery-Llama-3.2-1B" 

# Note: Please ensure you rotate your token since it was exposed in the logs!
login(token="[HF_TOKEN_REMOVED]") 

# Push Model
# This will now correctly resolve to "Joshua1702/fairrecovery-Llama-3.2-1B"
model.push_to_hub(
    f"{HF_USERNAME}/{MODEL_NAME}",
    commit_message="Fair-GRPO-RLVR trained on FairRecovery++ env"
)

# Push Tokenizer
tokenizer.push_to_hub(f"{HF_USERNAME}/{MODEL_NAME}")

print(f"✅ Published to HuggingFace Hub: https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

print("\nAdd this to your README.md:")
print(f"[![Model](https://img.shields.io/badge/🤗_Model-{MODEL_NAME}-orange)](https://huggingface.co/{HF_USERNAME}/{MODEL_NAME})")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


Saved model to https://huggingface.co/Joshua1702/fairrecovery-Llama-3.2-1B


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpjpay_b3_/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.


✅ Published to HuggingFace Hub: https://huggingface.co/Joshua1702/fairrecovery-Llama-3.2-1B

Add this to your README.md:
[![Model](https://img.shields.io/badge/🤗_Model-fairrecovery-Llama-3.2-1B-orange)](https://huggingface.co/Joshua1702/fairrecovery-Llama-3.2-1B)
